In [6]:
import asyncio
import pandas as pd
import numpy as np
import json
import os
import re
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from concurrent.futures import ThreadPoolExecutor, as_completed
import random
import time
from tqdm import tqdm
from typing import List, Dict, Optional

class OptimizedSeleniumScraper:
    def __init__(self, max_workers=8):
        self.max_workers = max_workers
        self.drivers = []
    
    def setup_driver(self):
        """Ultra-optimalizált driver beállítások"""
        options = webdriver.ChromeOptions()
        
        # Teljesítmény optimalizációk
        options.add_argument("--headless=new")  # Új headless mód
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--disable-extensions")
        options.add_argument("--disable-logging")
        options.add_argument("--disable-web-security")
        options.add_argument("--allow-running-insecure-content")
        
        # Gyorsítás - ELŐSZÖR TESZTELD JS NÉLKÜL!
        options.add_argument("--disable-images")
        # options.add_argument("--disable-javascript")  # Kommentezd ki, ha nem működik
        options.add_argument("--disable-plugins")
        options.add_argument("--disable-java")
        options.add_argument("--disable-background-timer-throttling")
        options.add_argument("--disable-backgrounding-occluded-windows")
        options.add_argument("--disable-renderer-backgrounding")
        
        # Memória optimalizáció
        options.add_argument("--memory-pressure-off")
        options.add_argument("--max_old_space_size=4096")
        
        # Hálózat optimalizáció
        options.add_argument("--aggressive-cache-discard")
        options.add_argument("--disable-background-networking")
        
        # Page load stratégia
        options.page_load_strategy = 'none'  # Nem vár a teljes betöltésre
        
        prefs = {
            "profile.managed_default_content_settings.images": 2,
            "profile.default_content_setting_values.notifications": 2,
            "profile.default_content_settings.popups": 0,
            "profile.managed_default_content_settings.media_stream": 2,
        }
        options.add_experimental_option("prefs", prefs)
        
        try:
            driver = webdriver.Chrome(
                service=Service(ChromeDriverManager().install()),
                options=options
            )
            driver.set_page_load_timeout(10)
            driver.implicitly_wait(2)
            return driver
        except Exception as e:
            print(f"Driver létrehozási hiba: {e}")
            return None
    
    def extract_number(self, text: str) -> Optional[float]:
        """Optimalizált számkinyerő"""
        if not text or pd.isna(text):
            return None
        text = str(text).replace("\xa0", "").replace(".", "").replace(" ", "")
        match = re.search(r'(\d+,?\d*)', text)
        return float(match.group(1).replace(",", ".")) if match else None
    
    def smart_wait(self, driver, selector, timeout=3):
        """Intelligens várakozás elemre"""
        try:
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
            return True
        except:
            return False
    
    def scrape_listing(self, url: str, driver) -> Optional[Dict]:
        """Egy hirdetés adatainak kinyerése optimalizált módszerrel"""
        try:
            # Gyors navigáció
            driver.get(url)
            
            # Minimális várakozás a kulcs elemre
            if not self.smart_wait(driver, 'h1[data-id="h1"]', timeout=5):
                return None
            
            listing_data = {"url": url}
            
            # Gyors adatkinyerés
            selectors = {
                "title": 'h1[data-id="h1"]',
                "price": 'div.fc-black-2.fs-32.fw-900',
                "area_m2": 'div[data-cy="advert-details-first-param"]',
                "floor": 'div.text-nowrap.fc-black-2.fs-20.fw-bold',
                "rooms": 'div[data-cy="advert-details-second-param"]',
                "location": 'button[data-cy="advert-map-map-btn"] span.fs-16'
            }
            
            # Minden elem egyszerre
            for field, selector in selectors.items():
                try:
                    element = driver.find_element(By.CSS_SELECTOR, selector)
                    text = element.text.strip()
                    
                    if field in ["price", "area_m2", "rooms"]:
                        listing_data[field] = self.extract_number(text)
                    else:
                        listing_data[field] = text
                except:
                    listing_data[field] = None
            
            # További tulajdonságok gyors kinyerése
            try:
                items = driver.find_elements(By.CSS_SELECTOR, 'div[data-cy="advert-details-param-list-item"]')
                for item in items[:10]:  # Max 10 extra tulajdonság
                    try:
                        key = item.find_element(By.CSS_SELECTOR, 'span').text.strip().replace(":", "")
                        val = item.find_element(By.CSS_SELECTOR, '.fw-bold').text.strip()
                        listing_data[key] = val
                    except:
                        continue
            except:
                pass
            
            listing_data["scrape_date"] = datetime.now().strftime('%Y-%m-%d')
            return listing_data
            
        except Exception as e:
            print(f"Hiba a(z) {url} feldolgozásakor: {str(e)}")
            return None

class CacheManager:
    def __init__(self, cache_file="optimized_scrape_cache.json"):
        self.cache_file = cache_file
    
    def load_cache(self) -> Dict:
        if os.path.exists(self.cache_file):
            with open(self.cache_file) as f:
                return json.load(f)
        return {"processed_urls": []}
    
    def save_cache(self, urls: List[str]):
        cache = self.load_cache()
        cache["processed_urls"].extend(urls)
        cache["processed_urls"] = list(set(cache["processed_urls"]))
        with open(self.cache_file, 'w') as f:
            json.dump(cache, f)

def process_batch_optimized(urls: List[str], existing_urls: set, max_workers: int = 8) -> List[Dict]:
    """Optimalizált párhuzamos feldolgozás"""
    scraper = OptimizedSeleniumScraper(max_workers)
    results = []
    
    # Csak új URL-ek
    urls_to_process = [url for url in urls if url not in existing_urls and not pd.isna(url)]
    
    if not urls_to_process:
        return []
    
    # Driver pool létrehozása
    drivers = []
    for _ in range(max_workers):
        driver = scraper.setup_driver()
        if driver:
            drivers.append(driver)
    
    if not drivers:
        print("❌ Nem sikerült driver-eket létrehozni!")
        return []
    
    print(f"🚀 {len(drivers)} driver létrehozva, {len(urls_to_process)} URL feldolgozása...")
    
    try:
        with ThreadPoolExecutor(max_workers=len(drivers)) as executor:
            # Feladatok szétosztása
            futures = []
            for i, url in enumerate(urls_to_process):
                driver = drivers[i % len(drivers)]
                future = executor.submit(scraper.scrape_listing, url, driver)
                futures.append(future)
                
                # Kis késleltetés a túlterhelés elkerülésére
                time.sleep(random.uniform(0.05, 0.15))
            
            # Eredmények gyűjtése progress bar-ral
            for future in tqdm(as_completed(futures), total=len(futures), desc="Feldolgozás"):
                result = future.result()
                if result:
                    results.append(result)
    
    finally:
        # Driver-ek bezárása
        for driver in drivers:
            try:
                driver.quit()
            except:
                pass
    
    return results

def main_optimized():
    """Fő optimalizált függvény"""
    start_time = time.time()
    
    try:
        # Adatok betöltése
        links_df_all = pd.read_csv("zenga_links.csv")
        output_path = "zenga_listings_details_optimized.csv"
        
        # Meglévő adatok
        if os.path.exists(output_path):
            existing_df = pd.read_csv(output_path)
            existing_urls = set(existing_df["url"].tolist())
        else:
            existing_df = pd.DataFrame()
            existing_urls = set()
        
        # Csak új URL-ek
        links_df = links_df_all[~links_df_all["url"].isin(existing_urls)]

        # Cache
        cache_manager = CacheManager()
        cache = cache_manager.load_cache()
        existing_urls.update(cache["processed_urls"])
        
        print(f"📋 Összesen {len(links_df_all)} URL")
        print(f"✅ {len(existing_df)} már feldolgozott")
        print(f"🆕 {len(links_df)} új URL")
        
        # Batch méret dinamikus beállítása
        total_new = len(links_df) - len(existing_urls)
        batch_size = min(200, max(50, total_new // 10))
        
        all_results = []
        urls_list = links_df["url"].tolist()
        
        # Batch-ekben feldolgozás
        for i in range(0, len(urls_list), batch_size):
            batch = urls_list[i:i+batch_size]
            print(f"\n🔄 Batch {i//batch_size + 1}/{(len(urls_list)-1)//batch_size + 1}")
            
            batch_results = process_batch_optimized(batch, existing_urls, max_workers=9)
            all_results.extend(batch_results)
            
            # Részeredmények mentése
            if all_results:
                new_df = pd.DataFrame(all_results)
                if not existing_df.empty:
                    updated_df = pd.concat([existing_df, new_df], ignore_index=True)
                else:
                    updated_df = new_df
                
                updated_df.to_csv(output_path, index=False)
                
                # Cache frissítés
                processed_urls = [item["url"] for item in all_results]
                cache_manager.save_cache(processed_urls)
                
                print(f"💾 {len(all_results)} új hirdetés mentve")
        
        # Végeredmény
        elapsed = time.time() - start_time
        rate = len(all_results) / (elapsed / 60) if elapsed > 0 else 0
        
        print(f"\n🎉 BEFEJEZVE!")
        print(f"⏱️  Idő: {elapsed/60:.1f} perc")
        print(f"📊 Új adatok: {len(all_results)}")
        print(f"🚀 Sebesség: {rate:.0f} hirdetés/perc")
        print(f"📈 Gyorsulás: {rate/36:.1f}x az eredetihez képest")
        
    except Exception as e:
        print(f"❌ Főprogram hiba: {str(e)}")

# Futtatás
if __name__ == "__main__":
    main_optimized()

📋 Összesen 440 URL
✅ 1961 már feldolgozott
🆕 -1521 új URL

🔄 Batch 1/9
🚀 9 driver létrehozva, 49 URL feldolgozása...


Feldolgozás: 100%|██████████| 49/49 [00:47<00:00,  1.04it/s]


💾 36 új hirdetés mentve

🔄 Batch 2/9
🚀 9 driver létrehozva, 50 URL feldolgozása...


Feldolgozás: 100%|██████████| 50/50 [00:36<00:00,  1.38it/s]


💾 36 új hirdetés mentve

🔄 Batch 3/9
🚀 9 driver létrehozva, 50 URL feldolgozása...


...

💾 230 új hirdetés mentve

🔄 Batch 8/9
🚀 9 driver létrehozva, 50 URL feldolgozása...


Feldolgozás: 100%|██████████| 50/50 [00:43<00:00,  1.16it/s]


💾 271 új hirdetés mentve

🔄 Batch 9/9
🚀 9 driver létrehozva, 40 URL feldolgozása...


Feldolgozás: 100%|██████████| 40/40 [00:34<00:00,  1.15it/s]


💾 305 új hirdetés mentve

🎉 BEFEJEZVE!
⏱️  Idő: 13.5 perc
📊 Új adatok: 305
🚀 Sebesség: 23 hirdetés/perc
📈 Gyorsulás: 0.6x az eredetihez képest
